In [17]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
import mlflow
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# load API keys from .env
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), '.env'))
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# initialize LLM - using llama3 via Groq 
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=GROQ_API_KEY,
    temperature=0.3
)

# embedding model - same one used in topic modeling for consistency
embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

print("GROQ_API_KEY loaded:", GROQ_API_KEY is not None)
print("LLM ready:", llm.model_name)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3288.47it/s]


GROQ_API_KEY loaded: True
LLM ready: llama-3.1-8b-instant


In [18]:
# UPDATED: load enriched dataset with NER entities and keywords
df = pd.read_csv("../data/processed/comments_enriched.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Sample entities:", df["entities"].iloc[1])
print("Sample keywords:", df["keywords"].iloc[1])

Shape: (27330, 20)
Columns: ['video_id', 'video_name', 'comment_id', 'author', 'text', 'likes', 'published_at', 'reply_count', 'text_clean', 'text_lemma', 'text_for_model', 'sentiment', 'score_negative', 'score_neutral', 'score_positive', 'confidence', 'topic', 'topic_name', 'entities', 'keywords']
Sample entities: []
Sample keywords: ['create destroying', 'don create', 'transform']


In [19]:
# get top 15 topics by size (exclude outliers)
top_topics = df[df["topic"] != -1]["topic_name"].value_counts().head(15).index.tolist()

topic_summaries = []

for topic_name in top_topics:
    topic_comments = df[df["topic_name"] == topic_name]["text"].tolist()
    
    # sample up to 30 comments to stay within token limits
    sample = topic_comments[:30]
    comments_text = "\n".join([f"- {c}" for c in sample])
    
    # structured prompt asking LLM to summarize the topic
    prompt = f"""You are analyzing YouTube comments about AI and the future of work.

Topic: {topic_name}
Number of comments in this topic: {len(topic_comments)}

Sample comments:
{comments_text}

Write a 3-4 sentence summary of what people in this topic cluster are discussing.
Include the general sentiment and key concerns or opinions expressed.
Be specific and grounded in the actual comments."""

    response = llm.invoke(prompt)
    
    topic_summaries.append({
        "topic_name": topic_name,
        "comment_count": len(topic_comments),
        "summary": response.content
    })
    
    print(f"Summarized: {topic_name} ({len(topic_comments)} comments)")

print(f"\nDone. {len(topic_summaries)} topic summaries generated.")

Summarized: 0_jobs_will_ai_to (2134 comments)
Summarized: 1_code_ai_it_use (1682 comments)
Summarized: 3_art_and_it_to (903 comments)
Summarized: 2_rate_faulure_music_10 (781 comments)
Summarized: 4_video_thank_videos_this (706 comments)
Summarized: 11_companies_they_corporations_company (635 comments)
Summarized: 6_robots_robot_will_and (614 comments)
Summarized: 5_capitalism_communism_marx_capitalist (604 comments)
Summarized: 18_rich_poor_people_will (520 comments)
Summarized: 20_tech_technology_companies_bros (490 comments)
Summarized: 7_llms_llm_are_models (483 comments)
Summarized: 8_income_universal_basic_tax (475 comments)
Summarized: 9_thank_beautiful_your_you (444 comments)
Summarized: 29_humans_replace_human_ai (424 comments)
Summarized: 10_slop_ai_is_it (423 comments)

Done. 15 topic summaries generated.


In [20]:
# initialize ChromaDB - persistent storage so we don't rebuild every run
chroma_client = chromadb.PersistentClient(path="../data/processed/chromadb")

# helper function to embed text using our sentence transformer
def embed_texts(texts):
    return embedder.encode(texts, show_progress_bar=True, batch_size=256).tolist()

print("ChromaDB initialized.")
print("Existing collections:", [c.name for c in chroma_client.list_collections()])

ChromaDB initialized.
Existing collections: ['transcripts', 'comments', 'topic_summaries']


In [21]:
# delete collection if it exists so we can rebuild clean
if "comments" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("comments")

comments_collection = chroma_client.create_collection(
    name="comments",
    metadata={"hnsw:space": "cosine"}  # cosine similarity for semantic search
)

# prepare data in batches to avoid memory issues
BATCH_SIZE = 500
total = len(df)

for start in range(0, total, BATCH_SIZE):
    end = min(start + BATCH_SIZE, total)
    batch = df.iloc[start:end]
    
    texts = batch["text"].tolist()
    embeddings = embed_texts(texts)
    
    # metadata fields - these enable filtered retrieval
    metadatas = [{
    "video_name":   str(row["video_name"]),
    "sentiment":    str(row["sentiment"]),
    "topic_name":   str(row["topic_name"]),
    "likes":        int(row["likes"]),
    "published_at": str(row["published_at"]),
    "confidence":   float(row["confidence"]),
    # NEW: entity and keyword metadata for richer filtering
    "entities":     str(row["entities"]),
    "keywords":     str(row["keywords"])
    } for _, row in batch.iterrows()]
    
    ids = [str(row["comment_id"]) for _, row in batch.iterrows()]
    
    comments_collection.add(
        embeddings=embeddings,
        documents=texts,
        metadatas=metadatas,
        ids=ids
    )
    
    print(f"Indexed {end}/{total} comments")

print(f"\nComments collection: {comments_collection.count()} documents")

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.09it/s]


Indexed 500/27330 comments


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.25it/s]


Indexed 1000/27330 comments


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.10it/s]


...

Indexed 26500/27330 comments


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.14it/s]


Indexed 27000/27330 comments


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.23it/s]


Indexed 27330/27330 comments

Comments collection: 27330 documents


In [22]:
# delete and recreate
if "transcripts" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("transcripts")

transcripts_collection = chroma_client.create_collection(
    name="transcripts",
    metadata={"hnsw:space": "cosine"}
)

# sliding window chunking - same approach as lab's chunking file
# chunk_size=500 chars, overlap=100 chars to preserve context across chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

all_chunks = []
all_metadatas = []
all_ids = []

for transcript in transcripts:
    chunks = splitter.split_text(transcript["transcript"])
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_metadatas.append({
            "video_name": transcript["video_name"],
            "video_id":   transcript["video_id"],
            "chunk_id":   i,
            "total_chunks": len(chunks)
        })
        all_ids.append(f"{transcript['video_id']}_chunk_{i}")

# embed and store all transcript chunks
chunk_embeddings = embed_texts(all_chunks)
transcripts_collection.add(
    embeddings=chunk_embeddings,
    documents=all_chunks,
    metadatas=all_metadatas,
    ids=all_ids
)

print(f"Transcripts collection: {transcripts_collection.count()} documents")
print(f"Total chunks from {len(transcripts)} videos")

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.95it/s]


Transcripts collection: 261 documents
Total chunks from 8 videos


In [23]:
# delete and recreate
if "topic_summaries" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("topic_summaries")

summaries_collection = chroma_client.create_collection(
    name="topic_summaries",
    metadata={"hnsw:space": "cosine"}
)

summary_texts = [s["summary"] for s in topic_summaries]
summary_embeddings = embed_texts(summary_texts)

summaries_collection.add(
    embeddings=summary_embeddings,
    documents=summary_texts,
    metadatas=[{
        "topic_name":    s["topic_name"],
        "comment_count": s["comment_count"]
    } for s in topic_summaries],
    ids=[f"summary_{i}" for i in range(len(topic_summaries))]
)

print(f"Topic summaries collection: {summaries_collection.count()} documents")
print("\nCollections in ChromaDB:")
for col in chroma_client.list_collections():
    print(f"  {col.name}")

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.84it/s]

Topic summaries collection: 15 documents

Collections in ChromaDB:
  comments
  topic_summaries
  transcripts


In [24]:
# RETRIEVAL STRATEGY 1: Semantic Search 
# finds comments most similar in meaning to the query
def semantic_retrieval(query, collection, k=5, filter_dict=None):
    query_embedding = embedder.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        where=filter_dict  # optional metadata filter
    )
    return results["documents"][0], results["metadatas"][0]


# RETRIEVAL STRATEGY 2: MMR (Maximal Marginal Relevance) 
# balances relevance with diversity - avoids returning 5 near-identical comments
# same approach as mmrRetrieval() in the lab's 5_Retrieval.py
def mmr_retrieval(query, collection, k=5, fetch_k=20, lambda_mult=0.7):
    query_embedding = embedder.encode(query).tolist()
    
    # fetch more candidates than needed
    candidates = collection.query(
        query_embeddings=[query_embedding],
        n_results=fetch_k
    )
    
    docs = candidates["documents"][0]
    embeddings_list = candidates["embeddings"]
    
    if embeddings_list is None:
        # fallback to semantic if embeddings not returned
        return docs[:k], candidates["metadatas"][0][:k]
    
    import numpy as np
    candidate_embeddings = np.array(embeddings_list[0])
    q_emb = np.array(query_embedding)
    
    selected_indices = []
    remaining = list(range(len(docs)))
    
    for _ in range(min(k, len(docs))):
        if not remaining:
            break
        if not selected_indices:
            # first pick: most relevant to query
            scores = candidate_embeddings[remaining] @ q_emb
            best = remaining[scores.argmax()]
        else:
            # subsequent picks: balance relevance vs diversity
            rel_scores = candidate_embeddings[remaining] @ q_emb
            sel_embs = candidate_embeddings[selected_indices]
            div_scores = (candidate_embeddings[remaining] @ sel_embs.T).max(axis=1)
            combined = lambda_mult * rel_scores - (1 - lambda_mult) * div_scores
            best = remaining[combined.argmax()]
        selected_indices.append(best)
        remaining.remove(best)
    
    selected_docs = [docs[i] for i in selected_indices]
    selected_meta = [candidates["metadatas"][0][i] for i in selected_indices]
    return selected_docs, selected_meta


# RETRIEVAL STRATEGY 3: Metadata Filtered 
# retrieve only comments matching specific metadata criteria
# e.g. only negative comments, only from a specific video
def metadata_filtered_retrieval(query, collection, filter_dict, k=5):
    query_embedding = embedder.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        where=filter_dict
    )
    return results["documents"][0], results["metadatas"][0]


# RETRIEVAL STRATEGY 4: Multi-Collection Hybrid 
# queries comments + transcripts + summaries and combines results
# gives the LLM context from all three sources
def hybrid_retrieval(query, k_each=3):
    results = {}
    
    # get from comments
    comment_docs, comment_meta = semantic_retrieval(query, comments_collection, k=k_each)
    results["comments"] = list(zip(comment_docs, comment_meta))
    
    # get from transcripts
    transcript_docs, transcript_meta = semantic_retrieval(query, transcripts_collection, k=k_each)
    results["transcripts"] = list(zip(transcript_docs, transcript_meta))
    
    # get from topic summaries
    summary_docs, summary_meta = semantic_retrieval(query, summaries_collection, k=2)
    results["summaries"] = list(zip(summary_docs, summary_meta))
    
    return results


print("Retrieval functions defined.")
print("Strategies: semantic, mmr, metadata_filtered, hybrid")

Retrieval functions defined.
Strategies: semantic, mmr, metadata_filtered, hybrid


In [25]:
test_query = "Will AI replace software developers and programmers?"

print("=" * 60)
print(f"QUERY: {test_query}")
print("=" * 60)

# Strategy 1: Semantic
print("\n── SEMANTIC RETRIEVAL ──")
sem_docs, sem_meta = semantic_retrieval(test_query, comments_collection, k=3)
for i, (doc, meta) in enumerate(zip(sem_docs, sem_meta)):
    print(f"\n[{i+1}] Sentiment: {meta['sentiment']} | Video: {meta['video_name']}")
    print(f"     {doc[:150]}...")

# Strategy 2: MMR
print("\n── MMR RETRIEVAL (diverse results) ──")
mmr_docs, mmr_meta = mmr_retrieval(test_query, comments_collection, k=3, fetch_k=20)
for i, (doc, meta) in enumerate(zip(mmr_docs, mmr_meta)):
    print(f"\n[{i+1}] Sentiment: {meta['sentiment']} | Video: {meta['video_name']}")
    print(f"     {doc[:150]}...")

# Strategy 3: Metadata filtered - only negative comments
print("\n── METADATA FILTERED (negative sentiment only) ──")
neg_docs, neg_meta = metadata_filtered_retrieval(
    test_query, comments_collection,
    filter_dict={"sentiment": {"$eq": "negative"}}, k=3
)
for i, (doc, meta) in enumerate(zip(neg_docs, neg_meta)):
    print(f"\n[{i+1}] Sentiment: {meta['sentiment']} | Video: {meta['video_name']}")
    print(f"     {doc[:150]}...")

# Strategy 4: Hybrid
print("\n── HYBRID RETRIEVAL (all collections) ──")
hybrid_results = hybrid_retrieval(test_query, k_each=2)
for source, items in hybrid_results.items():
    print(f"\n  [{source.upper()}]")
    for doc, meta in items:
        print(f"  - {doc[:120]}...")

QUERY: Will AI replace software developers and programmers?

── SEMANTIC RETRIEVAL ──

[1] Sentiment: negative | Video: replacing_developers_wrong
     AI will never replace software developers.  Only people who do not understand software development thinK so.  I'm coning from 30 years of developing s...

[2] Sentiment: negative | Video: replacing_developers_wrong
     AI is absolutely going to replace all the developers. But the tools currently available to the general public are quite a bit premature and are built ...

[3] Sentiment: neutral | Video: replacing_developers_wrong
     AI can replace rest of the software engineering roles  but not developers atleast few years...

── MMR RETRIEVAL (diverse results) ──

[1] Sentiment: negative | Video: replacing_developers_wrong
     AI will never replace software developers.  Only people who do not understand software development thinK so.  I'm coning from 30 years of developing s...

[2] Sentiment: negative | Video: replacing_developers_

In [26]:
from langchain_core.prompts import ChatPromptTemplate

# Q&A PROMPT 
# structured prompt tells LLM exactly its role, what data it has, and how to answer
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an AI analyst specializing in public opinion about AI and the future of work.
You answer questions based on real YouTube comments and video transcripts.
Always ground your answers in the provided context.
If the context doesn't contain enough information, say so clearly.
Never make up opinions or statistics not present in the context."""),
    ("human", """Question: {question}

Context from YouTube comments:
{comment_context}

Context from video transcripts:
{transcript_context}

Context from topic summaries:
{summary_context}

Provide a comprehensive answer based on the context above.
Include specific examples from the comments where relevant.""")
])

# SUMMARIZATION PROMPT 
summarization_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an AI analyst summarizing public opinion from YouTube comments.
Be objective and represent the range of views present in the data."""),
    ("human", """Summarize the public opinion on the following topic based on these comments:

Topic: {topic}
Comments:
{context}

Provide:
1. Overall sentiment (positive/negative/neutral balance)
2. Main arguments made
3. Key concerns or hopes expressed
4. Notable quotes or examples""")
])


def answer_question(question, strategy="hybrid"):
    """Full RAG pipeline: retrieve then generate"""
    
    if strategy == "hybrid":
        results = hybrid_retrieval(question, k_each=3)
        comment_context = "\n\n".join([doc for doc, _ in results["comments"]])
        transcript_context = "\n\n".join([doc for doc, _ in results["transcripts"]])
        summary_context = "\n\n".join([doc for doc, _ in results["summaries"]])
    else:
        docs, _ = semantic_retrieval(question, comments_collection, k=5)
        comment_context = "\n\n".join(docs)
        transcript_context = "N/A"
        summary_context = "N/A"
    
    chain = qa_prompt | llm
    response = chain.invoke({
        "question": question,
        "comment_context": comment_context,
        "transcript_context": transcript_context,
        "summary_context": summary_context
    })
    
    return response.content


def summarize_topic(topic_name):
    """Summarize public opinion on a specific topic"""
    docs, _ = metadata_filtered_retrieval(
        topic_name, comments_collection,
        filter_dict={"topic_name": {"$eq": topic_name}}, k=10
    )
    context = "\n\n".join(docs)
    chain = summarization_prompt | llm
    response = chain.invoke({"topic": topic_name, "context": context})
    return response.content


print("Generation functions defined.")

Generation functions defined.


In [27]:
# Test 1: Q&A with hybrid retrieval
print("=" * 60)
print("TEST 1: Q&A with Hybrid Retrieval")
print("=" * 60)
question1 = "What do people think about AI replacing software developers?"
answer1 = answer_question(question1, strategy="hybrid")
print(f"Q: {question1}\n")
print(f"A: {answer1}")

print("\n" + "=" * 60)
print("TEST 2: Q&A with Semantic Retrieval")
print("=" * 60)
question2 = "What are people's concerns about capitalism and AI automation?"
answer2 = answer_question(question2, strategy="semantic")
print(f"Q: {question2}\n")
print(f"A: {answer2}")

print("\n" + "=" * 60)
print("TEST 3: Sentiment-Filtered Q&A")
print("=" * 60)
question3 = "What are the most positive views about AI and the future of work?"
pos_docs, pos_meta = metadata_filtered_retrieval(
    question3, comments_collection,
    filter_dict={"sentiment": {"$eq": "positive"}}, k=5
)
context = "\n\n".join(pos_docs)
chain = qa_prompt | llm
answer3 = chain.invoke({
    "question": question3,
    "comment_context": context,
    "transcript_context": "N/A",
    "summary_context": "N/A"
})
print(f"Q: {question3}\n")
print(f"A: {answer3.content}")

print("\n" + "=" * 60)
print("TEST 4: Topic Summarization")
print("=" * 60)
summary = summarize_topic("5_capitalism_communism_marx_capitalist")
print(f"Topic: capitalism_communism_marx_capitalist\n")
print(summary)

TEST 1: Q&A with Hybrid Retrieval
Q: What do people think about AI replacing software developers?

A: Based on the provided context, people have mixed opinions about AI replacing software developers. Some commenters believe that AI will never replace software developers, while others think it will eventually replace them. 

One commenter with 30 years of software development experience thinks that AI is just a tool for developers and not a replacement. They mention that companies want a "version of snake oil" and that AI is just the latest iteration of this. Another commenter agrees that AI should be used as a tool for developers, not replace them.

However, some commenters are more pessimistic about the future of software development. One commenter thinks that AI will replace all developers, but notes that the current tools available to the general public are premature and built on the wrong principles. They also mention that AI that can replace all developers exists, but is not yet a

In [28]:
# MLflow tracing - logs every RAG call for monitoring and evaluation
# matches the lab's approach: mlflow.langchain.autolog()
mlflow.set_tracking_uri("sqlite:///../mlruns/mlflow.db")
mlflow.set_experiment("youtube-intelligence-engine")
mlflow.langchain.autolog()

# run a traced Q&A session - MLflow captures inputs, outputs, latency
test_questions = [
    "What do people think about AI replacing software developers?",
    "What are concerns about capitalism and AI automation?",
    "What positive views exist about AI and future of work?",
    "How do people feel about Universal Basic Income and AI?",
    "What do artists and creative workers think about AI?"
]

with mlflow.start_run(run_name="rag_pipeline_test"):
    mlflow.log_param("embedding_model", "all-MiniLM-L6-v2")
    mlflow.log_param("llm_model", "llama-3.1-8b-instant")
    mlflow.log_param("retrieval_strategy", "hybrid")
    mlflow.log_param("num_test_questions", len(test_questions))
    mlflow.log_param("comments_in_db", comments_collection.count())
    mlflow.log_param("transcripts_in_db", transcripts_collection.count())
    mlflow.log_param("summaries_in_db", summaries_collection.count())
    
    answers = []
    for i, question in enumerate(test_questions):
        print(f"Q{i+1}: {question}")
        answer = answer_question(question, strategy="hybrid")
        answers.append({"question": question, "answer": answer})
        print(f"A{i+1}: {answer[:150]}...\n")
    
    # log all Q&A pairs as artifact
    mlflow.log_dict({"qa_pairs": answers}, "qa_test_results.json")
    mlflow.log_metric("questions_answered", len(answers))
    
    print("MLflow run complete.")
    print("Run 'mlflow ui' in terminal to view dashboard at http://localhost:5000")

Q1: What do people think about AI replacing software developers?
A1: Based on the provided context, people have mixed opinions about AI replacing software developers. Some commenters believe that AI will never replace s...

Q2: What are concerns about capitalism and AI automation?
A2: Based on the provided context, concerns about capitalism and AI automation revolve around the potential for AI to exacerbate income inequality, displa...

Q3: What positive views exist about AI and future of work?
A3: Based on the provided context, there are several positive views about AI and the future of work. Some of these views include:

1. **Increased producti...

Q4: How do people feel about Universal Basic Income and AI?
A4: Based on the provided context, people have mixed opinions about Universal Basic Income (UBI) and its relationship with Artificial Intelligence (AI). S...

Q5: What do artists and creative workers think about AI?
A5: Based on the provided context, artists and creative workers h